#Import les biblio

In [195]:
import pandas as pd
import numpy  as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

from collections import Counter

import warnings
warnings.filterwarnings('ignore')

#Import data

In [196]:
data = {
    "Étudier": ["Oui","Oui","Non","Non","Oui","Non","Oui","Oui","Non","Non",
                 "Oui","Oui","Non","Non","Oui","Oui","Non","Oui","Non","Oui"],
    "Dormir":  ["Oui","Non","Oui","Non","Oui","Oui","Non","Oui","Non","Oui",
                 "Oui","Non","Oui","Non","Oui","Oui","Non","Non","Oui","Oui"],
    "Temps":   ["Long","Court","Court","Long","Long","Long","Court","Court","Court","Long",
                 "Long","Long","Court","Long","Court","Long","Court","Court","Long","Long"],
    "Réussite":["Oui","Oui","Non","Non","Oui","Non","Oui","Oui","Non","Non",
                 "Oui","Oui","Non","Non","Oui","Oui","Non","Oui","Non","Oui"]
}

df = pd.DataFrame(data)

In [197]:
#Transfer à 0 et 1 :
df = df.replace({'Oui': 1, 'Non': 0})
df['Temps']= df['Temps'].astype('category')
df['Temps'] = df['Temps'].cat.codes
df.head()

,Étudier,Dormir,Temps,Réussite
0,1,1,1,1
1,1,0,0,1
2,0,1,0,0
3,0,0,1,0
4,1,1,1,1


In [198]:
unique_counts = {}
for column in df.columns:
    unique_counts[column] = df[column].nunique()

print(unique_counts)

{'Étudier': 2, 'Dormir': 2, 'Temps': 2, 'Réussite': 2}


In [199]:
Y = 'Réussite'
unique_counts[Y]



2

#Naive Bayes

In [200]:
df_test_new = pd.DataFrame({
    "Étudier": [1, 0, 1, 0],
    "Dormir":  [0, 1, 1, 0],
    "Temps":   [1, 0, 0, 1],
    "Réussite":[1, 0, 1, 0]
})
X_test = df_test_new.drop(columns=[Y])
y_test = df_test_new[Y]
len(X_test)

4

In [201]:
def ProbaY(y):
  P=[]
  for i in range(unique_counts[Y]):
    p = len(df[df[Y] == df[Y].unique()[i]]) / len(df)
    P.append(p)
  return P

In [202]:
ProbaY('Réussite')

[0.55, 0.45]

In [203]:
#Proba de x= ?? tq y =1 :
def Proba_Xoui(y):
    P = []
    features = df.columns.tolist()
    features.remove(y)

    k = 2

    for feature in features:
        for val in [0, 1]:
            N_Xi_val_Y1 = len(df[(df[feature] == val) & (df[y] == 1)])
            N_Y1 = len(df[df[y] == 1])
            p = (N_Xi_val_Y1 + 1) / (N_Y1 + k)
            P.append(p)

    return P
Proba_Xoui(Y)

[0.07692307692307693,
 0.9230769230769231,
 0.38461538461538464,
 0.6153846153846154,
 0.46153846153846156,
 0.5384615384615384]

In [204]:
def Proba_Xno(y):
    P = []
    features = df.columns.tolist()
    features.remove(y)

    k = 2

    for feature in features:
        for val in [0, 1]:
            N_Xi_val_Y1 = len(df[(df[feature] == val) & (df[y] == 1)])
            N_Y1 = len(df[df[y] == 0])
            p = (N_Xi_val_Y1 + 1) / (N_Y1 + k)  # Laplace smoothing
            P.append(p)

    return P


In [205]:

ProbaOui = Proba_Xoui(Y)
ProbaNo = Proba_Xno(Y)
p = ProbaY(Y)
y1, y0 = p[0], p[1]

D ={
    "Étudier": {
        0: [ProbaNo[0],ProbaOui[0]],
        1: [ProbaNo[1],ProbaOui[1]]
    },
    "Dormir": {
        0: [ProbaNo[2],ProbaOui[2]],
        1: [ProbaNo[3],ProbaOui[3]]
    },
    "Temps": {
        0: [ProbaNo[4],ProbaOui[4]],
        1: [ProbaNo[5],ProbaOui[5]]
    }
}


In [215]:
def Naive_baise(X_test,y_test,D,y1,y0):
  y_pred = []
  for i in range(len(X_test)):
    prob_Y1 = np.log(y1)
    prob_Y0 = np.log(y0)
    for j in range(len(X_test.columns)):
      feature = X_test.columns[j]
      val = X_test.iloc[i, j]
      prob_Y1 += np.log(D[feature][val][0])
      prob_Y0 += np.log(D[feature][val][1])
    if prob_Y1 > prob_Y0:
        y_pred.append(1)
    else:
        y_pred.append(0)
  acc = accuracy_score(y_test, y_pred)

  return acc


In [216]:
acc = Naive_baise(X_test,y_test,D,y1,y0)
print(acc)

0.5


**1. 0.5 → le modèle prédit correctement 2 lignes sur 4.**

  - Avec si peu de données, ce n’est pas étonnant :

  - Ton train contient 20 lignes → très petit pour Naive Bayes.

  - Les features sont très corrélées, ce qui peut fausser les probabilités conditionnelles

**2. Pourquoi ce n’est pas alarmant**

- Sur un petit dataset, accuracy peut beaucoup varier.

- L’important est que la logique du code fonctionne et que tu multiplies bien les probabilités conditionnelles.